# Probabilistic Roadmap (PRM) – Bewegungsplanung für CPS

**Modul:** Cyber-Physische Systeme – Prognose und Planung

## Lernziel
Sie können den Probabilistic-Roadmap-Algorithmus anwenden und erklären, wie er den freien
Konfigurationsraum $C_{free}$ durch einen Graphen approximiert.

## Idee des Algorithmus
Den $C_{free}$-Raum feingranular zu diskretisieren führt zu riesigen Graphen (v. a. in hochdimensionalen
C-Spaces). Der PRM umgeht das durch **zufälliges Sampling**:

1. Ziehe einen zufälligen Punkt im Raum.
2. Liegt der Punkt in einem Hindernis? → verwerfen. Sonst: als **Knoten** in den Graphen aufnehmen.
3. Versuche den neuen Knoten mit seinen nächsten Nachbarn durch **Geraden (Kanten)** zu verbinden.
   Eine Kante wird nur eingefügt, wenn sie kein Hindernis schneidet.
4. Wiederhole, bis genug Knoten vorhanden sind.

Auf dem entstandenen Graphen kann anschließend ein Algorithmus für den kürzesten Pfad
(Dijkstra, A*) ausgeführt werden.

Das Notebook ist in vier Teile gegliedert: **(1)** Umgebung definieren, **(2)** PRM implementiert,
**(3)** Animation, **(4)** Pfadsuche. Zum Experimentieren sind Parameter-Zellen markiert mit ✏️.

In [ ]:
import heapq
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle

# Animation soll komplett im Notebook eingebettet werden dürfen
plt.rcParams["animation.embed_limit"] = 50  # MB

# Ein Hindernis ist ein Rechteck: (x_min, y_min, x_max, y_max)
Rect = Tuple[float, float, float, float]
Point = Tuple[float, float]

## 1. Die Umgebung

Der Raum ist ein 2-dimensionaler Bereich $[0, w] \times [0, h]$. Hindernisse werden als **Rechtecke**
über die Koordinaten `(x_min, y_min, x_max, y_max)` angegeben.

Die Klasse `Environment` kann zwei Dinge prüfen:
- `point_free`: Liegt ein Punkt im freien Raum?
- `segment_free`: Verläuft eine Strecke zwischen zwei Punkten komplett im freien Raum?
  (Hierfür wird der exakte Liang-Barsky-Test für Strecke gegen Rechteck verwendet.)

In [ ]:
class Environment:
    """Zweidimensionaler Raum mit rechteckigen Hindernissen.

    Attributes:
        width: Breite des Raumes.
        height: Höhe des Raumes.
        obstacles: Liste der Hindernisse als (x_min, y_min, x_max, y_max).
        margin: Sicherheitsabstand, um den Hindernisse virtuell vergrößert werden
            (entspricht dem Radius eines kreisförmigen Roboters).
    """

    def __init__(self, width: float = 10.0, height: float = 10.0,
                 obstacles: Optional[List[Rect]] = None, margin: float = 0.0):
        """Erzeugt eine Umgebung.

        Args:
            width: Breite des Raumes.
            height: Höhe des Raumes.
            obstacles: Liste von Rechtecken (x_min, y_min, x_max, y_max).
            margin: Sicherheitsabstand um Hindernisse herum.
        """
        self.width = width
        self.height = height
        self.obstacles = list(obstacles) if obstacles else []
        self.margin = margin

    def _inflated(self):
        """Gibt die um `margin` vergrößerten Hindernisse zurück."""
        m = self.margin
        return [(x0 - m, y0 - m, x1 + m, y1 + m) for x0, y0, x1, y1 in self.obstacles]

    def point_free(self, p: Point) -> bool:
        """Prüft, ob ein Punkt im freien Raum liegt.

        Args:
            p: Punkt (x, y).

        Returns:
            True, wenn der Punkt innerhalb des Raumes und außerhalb aller Hindernisse liegt.
        """
        x, y = p
        if not (0 <= x <= self.width and 0 <= y <= self.height):
            return False
        return not any(x0 <= x <= x1 and y0 <= y <= y1 for x0, y0, x1, y1 in self._inflated())

    def segment_free(self, p: Point, q: Point) -> bool:
        """Prüft, ob die Strecke von p nach q kein Hindernis schneidet.

        Verwendet den Liang-Barsky-Algorithmus (Clipping der Strecke an jedem Rechteck).

        Args:
            p: Startpunkt (x, y).
            q: Endpunkt (x, y).

        Returns:
            True, wenn die Strecke vollständig im freien Raum verläuft.
        """
        (px, py), (qx, qy) = p, q
        dx, dy = qx - px, qy - py
        for x0, y0, x1, y1 in self._inflated():
            t0, t1 = 0.0, 1.0
            # Jede Rechteckseite liefert eine Bedingung p_i * t <= q_i
            for pi, qi in ((-dx, px - x0), (dx, x1 - px), (-dy, py - y0), (dy, y1 - py)):
                if pi == 0:
                    if qi < 0:        # parallel und außerhalb
                        t0, t1 = 1.0, 0.0
                        break
                else:
                    t = qi / pi
                    if pi < 0:
                        t0 = max(t0, t)
                    else:
                        t1 = min(t1, t)
            if t0 <= t1:              # Strecke liegt (teilweise) im Rechteck
                return False
        return True

    def draw(self, ax: plt.Axes) -> None:
        """Zeichnet Raum und Hindernisse in eine Matplotlib-Achse.

        Args:
            ax: Die Achse, in die gezeichnet wird.
        """
        ax.set_xlim(0, self.width)
        ax.set_ylim(0, self.height)
        ax.set_aspect("equal")
        for x0, y0, x1, y1 in self.obstacles:
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, color="black", zorder=1))

### ✏️ Hindernisse vorgeben

Standardmäßig wird die Welt aus der Vorlesung erzeugt: ein **schmaler, S-förmiger Gang** zwischen den
Hindernissen. Ändern Sie die Liste `OBSTACLES`, um eigene Welten zu bauen – jedes Hindernis ist ein
Tupel `(x_min, y_min, x_max, y_max)`.

In [ ]:
WIDTH, HEIGHT = 10, 10

OBSTACLES = [
    (3.0, 8.0, 7.0, 10.0),   # oberer Block
    (5.3, 2.0, 7.0, 8.0),    # rechte Säule
    (3.0, 1.0, 4.8, 7.0),    # linke Säule
    (3.0, 0.0, 7.0, 1.0),    # unterer Block
]

env = Environment(WIDTH, HEIGHT, OBSTACLES)

fig, ax = plt.subplots(figsize=(5, 5))
env.draw(ax)
ax.set_title("Umgebung mit Hindernissen")
plt.show()

## 2. Der PRM-Algorithmus

Die Klasse `PRM` baut den Graphen auf. Parameter:

| Parameter | Bedeutung |
|---|---|
| `n_nodes` | Anzahl der Knoten, die im freien Raum erzeugt werden |
| `k` | Jeder neue Knoten wird mit höchstens `k` nächsten Nachbarn verbunden |
| `max_dist` | Nachbarn, die weiter weg sind, werden nicht verbunden |
| `seed` | Startwert des Zufallsgenerators (gleicher Seed → gleiches Ergebnis) |

Beim Aufbau wird zusätzlich ein **Ereignisprotokoll** (`events`) geführt: `("node", i)` bzw.
`("edge", i, j)`. Damit lässt sich der Aufbau später Schritt für Schritt animieren.

In [ ]:
class PRM:
    """Probabilistic Roadmap Planer für einen zweidimensionalen Raum.

    Attributes:
        env: Die Umgebung, in der geplant wird.
        n_nodes: Gewünschte Anzahl an Knoten.
        k: Maximale Anzahl an Nachbarn, mit denen ein neuer Knoten verbunden wird.
        max_dist: Maximale Länge einer Kante.
        nodes: Liste der Knotenkoordinaten (x, y).
        edges: Liste der Kanten als Indexpaare (i, j).
        events: Aufbau-Protokoll, Einträge ("node", i) oder ("edge", i, j).
        n_rejected: Anzahl der verworfenen Samples (lagen in einem Hindernis).
    """

    def __init__(self, env: Environment, n_nodes: int = 150, k: int = 10,
                 max_dist: float = 3.0, seed: Optional[int] = 42):
        """Erzeugt einen PRM-Planer (der Graph wird erst mit `build` aufgebaut).

        Args:
            env: Die Umgebung.
            n_nodes: Anzahl der Knoten im freien Raum.
            k: Maximale Anzahl Nachbarn pro neuem Knoten.
            max_dist: Maximale Kantenlänge.
            seed: Seed für den Zufallsgenerator (None für nicht reproduzierbar).
        """
        self.env = env
        self.n_nodes = n_nodes
        self.k = k
        self.max_dist = max_dist
        self.rng = np.random.default_rng(seed)
        self.nodes: List[Point] = []
        self.edges: List[Tuple[int, int]] = []
        self.events: list = []
        self.n_rejected = 0

    def build(self) -> "PRM":
        """Baut die Roadmap auf.

        Für jeden zufälligen Punkt im Raum: Liegt er im freien Raum, wird er Knoten und
        mit seinen nächsten Nachbarn verbunden, sofern die Verbindungsgerade frei ist.

        Returns:
            Das PRM-Objekt selbst (für Method Chaining).
        """
        self.nodes, self.edges, self.events, self.n_rejected = [], [], [], 0
        while len(self.nodes) < self.n_nodes:
            # 1. Zufälligen Punkt ziehen
            p = (self.rng.uniform(0, self.env.width), self.rng.uniform(0, self.env.height))
            # 2. In Hindernis? -> verwerfen
            if not self.env.point_free(p):
                self.n_rejected += 1
                continue
            # 3. Als Knoten aufnehmen
            new = self._add_node(p)
            # 4. Mit den nächsten Nachbarn verbinden
            for j in self._nearest(p, exclude=new):
                if self.env.segment_free(p, self.nodes[j]):
                    self.edges.append((new, j))
                    self.events.append(("edge", new, j))
        return self

    def _add_node(self, p: Point) -> int:
        """Fügt einen Knoten hinzu und protokolliert das Ereignis."""
        self.nodes.append(p)
        self.events.append(("node", len(self.nodes) - 1))
        return len(self.nodes) - 1

    def _nearest(self, p: Point, exclude: int) -> List[int]:
        """Indizes der bis zu k nächsten Knoten innerhalb von max_dist (aufsteigend sortiert)."""
        if len(self.nodes) <= 1:
            return []
        pts = np.asarray(self.nodes)
        d = np.linalg.norm(pts - np.asarray(p), axis=1)
        d[exclude] = np.inf
        order = np.argsort(d)[: self.k]
        return [int(i) for i in order if d[i] <= self.max_dist]

    def n_components(self) -> int:
        """Anzahl der zusammenhängenden Teilgraphen (Komponenten) der Roadmap.

        Returns:
            Anzahl der Komponenten. Bei 1 ist die Roadmap vollständig zusammenhängend.
        """
        parent = list(range(len(self.nodes)))

        def find(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        for i, j in self.edges:
            parent[find(i)] = find(j)
        return len({find(i) for i in range(len(self.nodes))})

    def draw(self, ax: plt.Axes, show_graph: bool = True) -> None:
        """Zeichnet Umgebung und (optional) die fertige Roadmap.

        Args:
            ax: Matplotlib-Achse.
            show_graph: Wenn True, werden Knoten und Kanten gezeichnet.
        """
        self.env.draw(ax)
        if show_graph and self.nodes:
            segs = [[self.nodes[i], self.nodes[j]] for i, j in self.edges]
            ax.add_collection(LineCollection(segs, colors="tab:blue", linewidths=0.8, zorder=2))
            xs, ys = zip(*self.nodes)
            ax.scatter(xs, ys, s=12, color="red", zorder=3)

### Roadmap einmal komplett aufbauen

In [ ]:
prm = PRM(env, n_nodes=300, k=10, max_dist=3.0, seed=42).build()

print(f"Knoten:               {len(prm.nodes)}")
print(f"Kanten:               {len(prm.edges)}")
print(f"Verworfene Samples:   {prm.n_rejected}")
print(f"Zusammenhangskomp.:   {prm.n_components()}")

fig, ax = plt.subplots(figsize=(6, 6))
prm.draw(ax)
ax.set_title("Fertige Roadmap")
plt.show()

## 3. Animation: Wie wächst die Roadmap?

Die Animation spielt das Ereignisprotokoll ab: rote Punkte sind Knoten, blaue Linien sind Kanten.
Mit `events_per_frame` steuern Sie die Geschwindigkeit (kleiner = langsamer, detaillierter).

### ✏️ Parameter zum Experimentieren

In [ ]:
N_NODES = 300        # Anzahl Knoten (probieren Sie auch 100-150: dann bleibt der Gang oft unverbunden)
K = 10               # max. Nachbarn pro Knoten
MAX_DIST = 3.0       # max. Kantenlänge
SEED = 42            # anderen Wert wählen -> anderes Ergebnis
EVENTS_PER_FRAME = 30 # Knoten/Kanten, die pro Animationsbild hinzukommen

In [ ]:
def animate_prm(prm: PRM, events_per_frame: int = 30, interval: int = 80) -> FuncAnimation:
    """Erzeugt eine Animation, die den Aufbau der Roadmap Schritt für Schritt zeigt.

    Args:
        prm: Ein bereits aufgebautes PRM-Objekt (`prm.build()` wurde aufgerufen).
        events_per_frame: Anzahl Ereignisse (Knoten oder Kanten) pro Bild.
        interval: Dauer eines Bildes in Millisekunden.

    Returns:
        Die Matplotlib-Animation (anzeigen mit `HTML(anim.to_jshtml())`).
    """
    # Für jedes Ereignis: wie viele Knoten/Kanten gibt es danach?
    n_nodes_after, n_edges_after, nn, ne = [], [], 0, 0
    for ev in prm.events:
        nn += ev[0] == "node"
        ne += ev[0] == "edge"
        n_nodes_after.append(nn)
        n_edges_after.append(ne)

    fig, ax = plt.subplots(figsize=(5, 5), dpi=70)
    prm.env.draw(ax)
    edge_lines = LineCollection([], colors="tab:blue", linewidths=0.8, zorder=2)
    ax.add_collection(edge_lines)
    node_plot = ax.scatter([], [], s=12, color="red", zorder=3)
    title = ax.set_title("")

    all_segs = [[prm.nodes[i], prm.nodes[j]] for i, j in prm.edges]
    pts = np.asarray(prm.nodes)

    last = len(prm.events) - 1
    frame_ids = list(range(0, last + 1, events_per_frame))
    if frame_ids[-1] != last:
        frame_ids.append(last)
    frame_ids += [last] * 10   # am Ende kurz stehen bleiben

    def update(frame):
        e = frame_ids[frame]
        nn, ne = n_nodes_after[e], n_edges_after[e]
        node_plot.set_offsets(pts[:nn])
        edge_lines.set_segments(all_segs[:ne])
        title.set_text(f"Knoten: {nn}   Kanten: {ne}")
        return node_plot, edge_lines, title

    anim = FuncAnimation(fig, update, frames=len(frame_ids), interval=interval, blit=False)
    plt.close(fig)   # verhindert ein zusätzliches statisches Bild
    return anim

In [ ]:
prm = PRM(env, n_nodes=N_NODES, k=K, max_dist=MAX_DIST, seed=SEED).build()
anim = animate_prm(prm, events_per_frame=EVENTS_PER_FRAME)
HTML(anim.to_jshtml())

**Beobachtungen / Fragen zur Diskussion:**

- Wie viele Zusammenhangskomponenten hat die Roadmap (`prm.n_components()`)? Setzen Sie `N_NODES = 150`:
  Warum ist die Roadmap nun **nicht** zusammenhängend, obwohl es einen Weg durch den Gang gibt?
- Dies ist das bekannte **Narrow-Passage-Problem**: Der Gang ist schmal, daher fallen nur wenige
  zufällige Samples hinein. Mit mehr Knoten oder größerem `MAX_DIST` wird die Roadmap verbunden.
- Wie verändern `K` und `MAX_DIST` die Anzahl der Kanten und die Rechenzeit?
- Bauen Sie eine eigene Welt (z. B. ein großes Hindernis in der Mitte, mehrere Räume mit Türen).

## 4. Pfadsuche auf der Roadmap

Ist die Roadmap aufgebaut, wird die Planung zu einer **Suche auf einem gewichteten Graphen**
(Kantengewicht = Länge). Wir verbinden Start und Ziel mit der Roadmap und suchen mit dem
Algorithmus von Dijkstra den kürzesten Pfad.

### ✏️ Start und Ziel

In [ ]:
START = (1.0, 9.0)
GOAL = (9.0, 1.0)

In [ ]:
def find_path(prm: PRM, start: Point, goal: Point) -> Optional[List[Point]]:
    """Sucht mit Dijkstra den kürzesten Pfad von start nach goal auf der Roadmap.

    Start und Ziel werden mit den bis zu `prm.k` nächsten sichtbaren Roadmap-Knoten verbunden.

    Args:
        prm: Aufgebaute Roadmap.
        start: Startpunkt (x, y).
        goal: Zielpunkt (x, y).

    Returns:
        Liste von Punkten von Start bis Ziel oder None, wenn kein Pfad existiert.
    """
    if not (prm.env.point_free(start) and prm.env.point_free(goal)):
        print("Start oder Ziel liegt in einem Hindernis!")
        return None

    nodes = prm.nodes + [start, goal]
    s, g = len(nodes) - 2, len(nodes) - 1
    pts = np.asarray(prm.nodes)

    # Adjazenzliste mit Kantengewichten (euklidische Länge)
    adj = {i: [] for i in range(len(nodes))}
    for i, j in prm.edges:
        w = float(np.linalg.norm(pts[i] - pts[j]))
        adj[i].append((j, w))
        adj[j].append((i, w))
    for extra in (s, g):
        d = np.linalg.norm(pts - np.asarray(nodes[extra]), axis=1)
        for j in np.argsort(d)[: prm.k]:
            if d[j] <= prm.max_dist and prm.env.segment_free(nodes[extra], nodes[j]):
                adj[extra].append((int(j), float(d[j])))
                adj[int(j)].append((extra, float(d[j])))

    # Dijkstra
    dist = {s: 0.0}
    prev = {}
    queue = [(0.0, s)]
    while queue:
        d_u, u = heapq.heappop(queue)
        if u == g:
            break
        if d_u > dist.get(u, np.inf):
            continue
        for v, w in adj[u]:
            if d_u + w < dist.get(v, np.inf):
                dist[v] = d_u + w
                prev[v] = u
                heapq.heappush(queue, (d_u + w, v))

    if g not in dist:
        return None
    path, u = [nodes[g]], g
    while u != s:
        u = prev[u]
        path.append(nodes[u])
    return path[::-1]

In [ ]:
path = find_path(prm, START, GOAL)

fig, ax = plt.subplots(figsize=(6, 6))
prm.draw(ax)
ax.plot(*START, "go", markersize=10, zorder=5, label="Start")
ax.plot(*GOAL, "m*", markersize=14, zorder=5, label="Ziel")
if path:
    length = sum(np.linalg.norm(np.subtract(a, b)) for a, b in zip(path, path[1:]))
    ax.plot(*zip(*path), color="orange", linewidth=3, zorder=4, label=f"Pfad (Länge {length:.2f})")
else:
    print("Kein Pfad gefunden – Roadmap nicht zusammenhängend zwischen Start und Ziel. "
          "Mehr Knoten oder größeres MAX_DIST probieren!")
ax.legend(loc="lower left")
plt.show()

## Zusammenfassung

- Der PRM approximiert $C_{free}$ durch einen Graphen aus **zufällig gesampelten Knoten** und
  **kollisionsfreien Kanten**.
- Der Aufwand hängt kaum von der Dimension des C-Space ab – deshalb eignet sich der PRM auch für Roboterarme
  mit vielen Freiheitsgraden.
- Der Algorithmus ist **probabilistisch vollständig**: Mit steigender Knotenzahl findet er (falls vorhanden) einen Pfad –
  aber bei schmalen Passagen braucht es sehr viele Samples.
- Verwandt: **RRT** (Rapidly Exploring Random Tree) wächst gezielt in unerforschte Bereiche (Voronoi Bias).

**Zum Weiterdenken:** Wie könnte man mehr Samples in den schmalen Gang bringen? (Stichwort: *Bridge Test*, *Gaussian Sampling*)